# 🐖 11 | Confiabilidad geométrica, detección y contexto de solapamiento

**RECURSO: CPU (sin GPU).** Este notebook NO entrena ni hace inferencia: utiliza los JSON de máscaras COCO del **Notebook 09**, evaluados ya con un test congelado por fecha. Mantiene las máscaras de referencia, los modelos, sus pesos y la configuración originales.

**Preguntas:** ¿Qué tan fiables son el área, el contorno y la localización geométrica de las máscaras correctamente emparejadas? ¿Qué errores de conteo y detección aparecen? ¿Cómo varían entre fechas y en escenas congestionadas?

**Criterios:** emparejamiento voraz por confianza, un GT por detección; IoU de máscara ≥ **0,50**, confianza ≥ **0,25** (umbral fijo descriptivo; *no es COCO AP*). El F1 de contornos usa **2 píxeles** de tolerancia. La superposición de **cajas de referencia** es solo un *proxy de congestión espacial*, **NO una etiqueta validada de oclusión real**. Sin calibración, no se expresan medidas en cm.

**Salidas:** Excel detallado (por instancia, imagen y fecha), figuras, reporte JSON, puntos de control de avance y un ZIP pequeño para adjuntar en ChatGPT. Si se interrumpe, permite continuar imagen por imagen.
**Corrección de recuperación:** en el resumen se cambió el acceso a la columna `gt` a `g['gt']` para evitar conflicto con el método `DataFrame.gt`. Los puntos de control (`checkpoint_*.jsonl`) existentes se reutilizan: no se recalculan imágenes ya completadas.


## 1. Dependencias y montaje de Drive (CPU)
 Instala bibliotecas ligeras. **No selecciones GPU** en Colab para esta etapa.

In [ ]:
%pip -q install "pycocotools>=2.0.8,<3" "numpy>=1.24,<3" "pandas>=2,<3" openpyxl opencv-python-headless matplotlib


In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime,timezone
from collections import defaultdict
import hashlib, json, zipfile, platform, sys
import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt
from pycocotools import mask as maskutils

drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/Proyecto de investigación: Visión Artificial y Deep Learning en Cerdos 🐖')
RES=ROOT/'03_Resultados'; EXPORT=ROOT/'05_Exportaciones_ZIP'
REPORTS=RES/'05_Reportes'; METRICS=RES/'03_Metricas'; FIGURES=RES/'04_Figuras'
CACHE09=REPORTS/'CERDOS_09_COCO_Armonizado'
GT_PATH=CACHE09/'CERDOS_09_GT_test_coco.json'
PREDS={'YOLO11n-seg':CACHE09/'CERDOS_09_Pred_YOLO_coco.json',
       'Mask R-CNN ResNet50-FPN':CACHE09/'CERDOS_09_Pred_MaskRCNN_coco.json'}
PROTO09_PATH=CACHE09/'CERDOS_09_Protocolo.json'
REPORT09_PATH=REPORTS/'CERDOS_09_Armonizacion_COCO.json'
for p in [GT_PATH,*PREDS.values(),PROTO09_PATH,REPORT09_PATH]:
    if not p.is_file(): raise FileNotFoundError(f'Falta artefacto del notebook 09: {p}')
for p in [METRICS,FIGURES,REPORTS,EXPORT]:p.mkdir(exist_ok=True,parents=True)
def sha256_file(p):
    h=hashlib.sha256()
    with p.open('rb') as fp:
        for chunk in iter(lambda: fp.read(4*1024*1024),b''):h.update(chunk)
    return h.hexdigest()

gt=json.loads(GT_PATH.read_text(encoding='utf-8'))
p09=json.loads(PROTO09_PATH.read_text(encoding='utf-8'))
r09=json.loads(REPORT09_PATH.read_text(encoding='utf-8'))
assert r09.get('protocolo')==p09,'Protocolo 09 y reporte 09 no coinciden'
assert len(gt['images'])==373 and len(gt['annotations'])==4648,'Referencia GT inesperada'
assert len({str(i['file_name'])[:10] for i in gt['images']})==5,'Fechas de prueba inesperadas'
predictions={name:json.loads(p.read_text(encoding='utf-8')) for name,p in PREDS.items()}
image_ids={int(i['id']) for i in gt['images']}
assert all(set(int(a['image_id']) for a in rows)<=image_ids for rows in predictions.values())
print('✅ CPU | 373 imágenes de prueba | 4648 instancias de referencia | 5 fechas')
print('✅ Predicciones reutilizadas:',{k:len(v) for k,v in predictions.items()})


## 2. Protocolo fijo, comprobación de integridad y reanudación
 Los puntos de control se almacenan **por imagen completa**, para no perder trabajo si se corta Colab. Si cambian las predicciones o el protocolo, el notebook se detendrá antes de mezclar resultados.

In [ ]:
EXP_DIR=REPORTS/'CERDOS_11_Geometria'
EXP_DIR.mkdir(parents=True,exist_ok=True)
PROTO_PATH=EXP_DIR/'CERDOS_11_Protocolo.json'
OUTPUT_XLSX=METRICS/'CERDOS_11_Confiabilidad_Geometrica.xlsx'
OUTPUT_FIG=FIGURES/'CERDOS_11_Geometria_Comparada.png'
OUTPUT_JSON=REPORTS/'CERDOS_11_Confiabilidad_Geometrica.json'
MODEL_KEYS={'YOLO11n-seg':'YOLO','Mask R-CNN ResNet50-FPN':'MaskRCNN'}
PROTO={
    'pipeline':'CERDOS_11_geometria_v1',
    'fuente':'Predicciones y GT COCO del notebook 09, test original ya evaluado',
    'img_count':373,'gt_count':4648,'score_min':0.25,'mask_iou_min':0.50,
    'boundary_tolerance_px':2,'gt_bbox_overlap_iou_proxy':0.10,
    'match':'predicciones por score descendente, emparejamiento voraz uno a uno por IoU',
    'hash_proto09':sha256_file(PROTO09_PATH),'hash_gt':sha256_file(GT_PATH),
    'hash_pred_yolo':sha256_file(PREDS['YOLO11n-seg']),
    'hash_pred_maskrcnn':sha256_file(PREDS['Mask R-CNN ResNet50-FPN']),
}
if PROTO_PATH.exists():
    if json.loads(PROTO_PATH.read_text(encoding='utf-8'))!=PROTO:
        raise RuntimeError('Hay un análisis 11 anterior con archivos o parámetros diferentes. NO mezclar.')
else:
    if OUTPUT_XLSX.exists() or OUTPUT_JSON.exists() or any(EXP_DIR.glob('checkpoint_*.jsonl')):
        raise RuntimeError('Hay resultados 11 anteriores sin protocolo. Revisa Drive antes de continuar.')
    PROTO_PATH.write_text(json.dumps(PROTO,indent=2,ensure_ascii=False),encoding='utf-8')

gt_by_img=defaultdict(list)
for a in gt['annotations']:gt_by_img[int(a['image_id'])].append(a)
pred_by_img={}
for model,rows in predictions.items():
    p=defaultdict(list)
    for r in rows:
        if float(r['score'])>=PROTO['score_min']:p[int(r['image_id'])].append(r)
    pred_by_img[model]=p
images=sorted(gt['images'],key=lambda x:int(x['id']))
assert len({i['id'] for i in images})==373
print('✅ Archivos y parámetros fijados. Nunca se modifican predicciones originales.')


## 3. Métricas geométricas y emparejamiento de instancias
 Solo calculamos Dice, IoU, área, distancia entre centroides y boundary-F1 para pares emparejados. Las detecciones sin par son falsos positivos; referencias sin par, falsos negativos. Estos valores descriptivos **no sustituyen** el AP de COCOeval del Notebook 09.

In [ ]:
def bbox_iou(b1,b2):
    x1=max(b1[0],b2[0]);y1=max(b1[1],b2[1])
    x2=min(b1[0]+b1[2],b2[0]+b2[2])
    y2=min(b1[1]+b1[3],b2[1]+b2[3])
    inter=max(0.,x2-x1)*max(0.,y2-y1)
    a1=max(0.,b1[2])*max(0.,b1[3]);a2=max(0.,b2[2])*max(0.,b2[3])
    return float(inter/(a1+a2-inter)) if a1+a2-inter>0 else 0.

def crowded_gt_flags(anns):
    out=[]
    for i,a in enumerate(anns):
        overlap=max((bbox_iou(a['bbox'],b['bbox']) for j,b in enumerate(anns) if j!=i),default=0.)
        out.append(overlap)
    return out

KERNEL=np.ones((3,3),np.uint8)
def boundary_f1(mask_a,mask_b,tol=2):
    # Gradiente morfológico y tolerancia simétrica de 2 píxeles.
    a=(mask_a>0).astype(np.uint8)
    b=(mask_b>0).astype(np.uint8)
    ea=cv2.morphologyEx(a,cv2.MORPH_GRADIENT,KERNEL)>0
    eb=cv2.morphologyEx(b,cv2.MORPH_GRADIENT,KERNEL)>0
    na=int(ea.sum());nb=int(eb.sum())
    if na==0 or nb==0:return np.nan
    db=cv2.dilate(eb.astype(np.uint8),cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(2*tol+1,2*tol+1)))>0
    da=cv2.dilate(ea.astype(np.uint8),cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(2*tol+1,2*tol+1)))>0
    precision=float(np.logical_and(ea,db).sum()/na)
    recall=float(np.logical_and(eb,da).sum()/nb)
    return 2*precision*recall/(precision+recall) if (precision+recall)>0 else 0.

def mask_geometry(gt_rle,pred_rle,shape):
    gr=maskutils.decode(gt_rle)>0
    pr=maskutils.decode(pred_rle)>0
    assert gr.shape==shape and pr.shape==shape,'Máscaras COCO con dimensiones inesperadas'
    ag=int(np.count_nonzero(gr));ap=int(np.count_nonzero(pr))
    inter=int(np.logical_and(gr,pr).sum())
    union=ag+ap-inter
    yg,xg=np.nonzero(gr);yp,xp=np.nonzero(pr)
    center=float(np.hypot(xg.mean()-xp.mean(),yg.mean()-yp.mean())) if ag and ap else np.nan
    return dict(mask_iou=float(inter/union) if union else np.nan,
                dice=float(2*inter/(ag+ap)) if ag+ap else np.nan,
                gt_area_px2=ag,pred_area_px2=ap,
                error_area_firmado_pct=float(100*(ap-ag)/ag) if ag else np.nan,
                error_area_abs_pct=float(100*abs(ap-ag)/ag) if ag else np.nan,
                error_centroide_px=center,
                error_centroide_pct_diagonal=float(100*center/np.hypot(*shape)) if np.isfinite(center) else np.nan,
                boundary_f1_2px=boundary_f1(gr,pr,PROTO['boundary_tolerance_px']))

def inspect_image(info, model):
    image_id=int(info['id']);w=int(info['width']);h=int(info['height']);date=str(info['file_name'])[:10]
    original=gt_by_img[image_id];scored=sorted(pred_by_img[model].get(image_id,[]),key=lambda a:float(a['score']),reverse=True)
    overlaps=crowded_gt_flags(original)
    n=len(original);m=len(scored)
    if n and m:
        matrix=np.asarray(maskutils.iou([p['segmentation'] for p in scored],
                                       [g['segmentation'] for g in original],[0]*n),dtype=float)
        assert matrix.shape==(m,n)
    else:matrix=np.zeros((m,n),float)
    gt_used=set();pred_used=set();pairs=[]
    for pi,pred in enumerate(scored):
        options=[gi for gi in range(n) if gi not in gt_used]
        if not options:break
        chosen=max(options,key=lambda gi:matrix[pi,gi])
        if matrix[pi,chosen] < PROTO['mask_iou_min']:continue
        gt_used.add(chosen);pred_used.add(pi)
        ann=original[chosen]
        geom=mask_geometry(ann['segmentation'],pred['segmentation'],(h,w))
        pairs.append({'modelo':model,'fecha':date,'image_id':image_id,
                      'imagen':str(info['file_name']),'gt_annotation_id':int(ann['id']),
                      'confianza_pred':float(pred['score']),
                      'solapamiento_max_bbox_gt':overlaps[chosen],
                      'proxy_bbox_congestionado':overlaps[chosen]>=PROTO['gt_bbox_overlap_iou_proxy'],
                      **geom})
    tp=len(pairs);fp=m-tp;fn=n-tp
    return {'img':{'modelo':model,'fecha':date,'image_id':image_id,'imagen':str(info['file_name']),
                   'gt':n,'pred_umbral':m,'TP':tp,'FP':fp,'FN':fn,
                   'precision':tp/(tp+fp) if tp+fp else np.nan,
                   'recall':tp/(tp+fn) if tp+fn else np.nan,
                   'error_conteo_firmado':m-n,'error_conteo_absoluto':abs(m-n),
                   'gt_bbox_congestionados':int(sum(x>=PROTO['gt_bbox_overlap_iou_proxy'] for x in overlaps)),
                   'gt_bbox_solapamiento_medio':float(np.mean(overlaps)) if overlaps else 0.},
            'pairs':pairs}
print('✅ Definidas las métricas; ninguna usa centímetros, identidades ni etiquetas reales de oclusión.')


## 4. Procesar y guardar progresivamente los 373 casos por modelo
 Cada imagen terminada se escribe en un archivo de reanudación. Una segunda ejecución omite las imágenes que ya están completas.

In [ ]:
def previous_rows(file):
    records={}
    if not file.exists():return records
    for line_number,line in enumerate(file.read_text(encoding='utf-8').splitlines(),start=1):
        if not line.strip():continue
        try:obj=json.loads(line)
        except json.JSONDecodeError:
            raise RuntimeError(f'Checkpoint {file.name} dañado en línea {line_number}; revisar antes de continuar.')
        image_id=int(obj['img']['image_id'])
        if image_id in records:raise RuntimeError(f'Checkpoint duplicado: {file.name}, id {image_id}')
        records[image_id]=obj
    return records

completed={}
for model,key in MODEL_KEYS.items():
    file=EXP_DIR/f'checkpoint_{key}.jsonl'
    recorded=previous_rows(file)
    if any(i not in {int(z['id']) for z in images} for i in recorded):raise RuntimeError('Checkpoint con imágenes desconocidas')
    todo=[img for img in images if int(img['id']) not in recorded]
    print(f'➜ {model}: {len(recorded)} imágenes ya hechas; {len(todo)} pendientes')
    with file.open('a',encoding='utf-8') as handle:
        for counter,info in enumerate(todo,start=1):
            result=inspect_image(info,model)
            handle.write(json.dumps(result,ensure_ascii=False,allow_nan=True)+'\n')
            handle.flush()
            recorded[int(info['id'])]=result
            if counter%25==0 or counter==len(todo):
                print(f'   {model}: procesadas {len(recorded)}/{len(images)} imágenes')
    assert len(recorded)==373
    completed[model]=recorded
print('✅ Geometría evaluada. Si vuelve a ejecutar, se reutilizarán estos resultados.')


## 5. Resultados, análisis de congestión espacial y archivos Excel
 Se informan promedios geométricos **solo entre máscaras emparejadas**, tasas TP/FP/FN entre todas las máscaras y diferencias de área y conteo por imagen. No se infiere oclusión verdadera desde solapamiento de cajas.

In [ ]:
image_rows=[r['img'] for model in MODEL_KEYS for r in completed[model].values()]
paired_rows=[p for model in MODEL_KEYS for r in completed[model].values() for p in r['pairs']]
df_img=pd.DataFrame(image_rows).sort_values(['modelo','image_id']).reset_index(drop=True)
df_pair=pd.DataFrame(paired_rows).sort_values(['modelo','image_id','gt_annotation_id']).reset_index(drop=True)
assert len(df_img)==746 and set(df_img.modelo)==set(MODEL_KEYS)
assert len(df_pair)>0,'No hay pares de máscaras; revisar umbrales'

summary=[]
for model,g in df_img.groupby('modelo'):
    part=df_pair[df_pair['modelo']==model]
    tp=int(g.TP.sum());fp=int(g.FP.sum());fn=int(g.FN.sum())
    assert tp+fn==4648,'TP + FN no concuerda con las etiquetas de referencia'
    summary.append({'modelo':model,'imagenes':int(len(g)),'GT_total':int(g['gt'].sum()),
                   'pred_total_conf_025':int(g.pred_umbral.sum()),
                   'TP_iou_050':tp,'FP_iou_050':fp,'FN_iou_050':fn,
                   'precision_micro':tp/(tp+fp) if tp+fp else np.nan,
                   'recall_micro':tp/(tp+fn) if tp+fn else np.nan,
                   'MAE_conteo_imagen':float(g.error_conteo_absoluto.mean()),
                   'IoU_media_matches':float(part.mask_iou.mean()),
                   'Dice_media_matches':float(part.dice.mean()),
                   'error_area_abs_pct_medio_matches':float(part.error_area_abs_pct.mean()),
                   'boundary_F1_2px_medio_matches':float(part.boundary_f1_2px.mean())})
df_summary=pd.DataFrame(summary)
df_date=(df_img.groupby(['modelo','fecha'],as_index=False)
         .agg(imagenes=('image_id','size'),GT=('gt','sum'),TP=('TP','sum'),FP=('FP','sum'),FN=('FN','sum'),
              MAE_conteo=('error_conteo_absoluto','mean')))
df_date['precision_micro']=df_date.TP/(df_date.TP+df_date.FP).replace(0,np.nan)
df_date['recall_micro']=df_date.TP/(df_date.TP+df_date.FN).replace(0,np.nan)
df_cong=(df_pair.groupby(['modelo','proxy_bbox_congestionado'],as_index=False)
         .agg(pares=('image_id','size'),IoU_media=('mask_iou','mean'),
              Dice_media=('dice','mean'),error_area_abs_pct=('error_area_abs_pct','mean'),
              contorno_F1_2px=('boundary_f1_2px','mean')))

limitations=[
  'Las métricas geométricas describen pares emparejados; no representan errores de objetos no detectados.',
  'Emparejamiento por confianza>=0.25 e IoU>=0.50: TP/FP/FN descriptivos, no COCO AP.',
  'Oclusión real NO etiquetada: solapamiento IoU de cajas GT >=0.10 solo sirve como proxy de congestión.',
  'La métrica contorno F1 usa tolerancia fija de 2 píxeles; depende de resolución.',
  'Áreas en px², sin calibración métrica; no se estima peso corporal ni tamaño en centímetros.',
  'Test consultado en notebooks 04,07,09: no es evaluación externa nueva ni validación entre granjas.',
  'La comparación refleja dos presupuestos de entrenamiento desiguales (YOLO 60 y Mask R-CNN 12 épocas).',
]
with pd.ExcelWriter(OUTPUT_XLSX,engine='openpyxl') as writer:
    df_summary.to_excel(writer,'Resumen',index=False)
    df_date.to_excel(writer,'Por_Fecha',index=False)
    df_img.to_excel(writer,'Por_Imagen',index=False)
    df_pair.to_excel(writer,'Pares_Geometricos',index=False)
    df_cong.to_excel(writer,'Proxy_Congestion',index=False)
    pd.DataFrame([PROTO]).to_excel(writer,'Protocolo',index=False)
    pd.DataFrame({'limitaciones':limitations}).to_excel(writer,'Limitaciones',index=False)
print('✅ Excel geométrico:',OUTPUT_XLSX)
display(df_summary)


In [ ]:
fig,axs=plt.subplots(1,2,figsize=(13,4.7))
names=df_summary['modelo'].tolist()
axs[0].bar(names,df_summary['IoU_media_matches'])
axs[0].set(ylim=(0,1),title='IoU medio entre pares correctamente emparejados',ylabel='IoU')
axs[1].bar(names,df_summary['MAE_conteo_imagen'])
axs[1].set(title='Error absoluto medio de conteo por imagen',ylabel='Instancias / imagen')
for ax in axs:ax.tick_params(axis='x',labelrotation=10)
fig.suptitle('FaroPigSeg | Análisis geométrico descriptivo (test por fecha)')
fig.tight_layout();fig.savefig(OUTPUT_FIG,dpi=170,bbox_inches='tight');plt.show()

result={'etapa':'CERDOS_11_confiabilidad_geometrica','fecha_utc':datetime.now(timezone.utc).isoformat(),
        'protocolo':PROTO,'resumen':df_summary.replace({np.nan:None}).to_dict('records'),
        'por_fecha':df_date.replace({np.nan:None}).to_dict('records'),
        'proxy_congestion':df_cong.replace({np.nan:None}).to_dict('records'),
        'limitaciones':limitations,
        'interpretacion':'Medidas descriptivas en pares emparejados, no AP ni prueba experimental de oclusión.'}
OUTPUT_JSON.write_text(json.dumps(result,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
print('✅ Reporte:',OUTPUT_JSON)


## 6. Exportación automática ZIP al computador y Drive

In [ ]:
from google.colab import files
ZIP=EXPORT/'CERDOS_11_Confiabilidad_Geometrica_Resultados.zip'
entries=[OUTPUT_XLSX,OUTPUT_FIG,OUTPUT_JSON,PROTO_PATH]
assert all(p.is_file() for p in entries),'Hay productos 11 faltantes; no crear ZIP'
with zipfile.ZipFile(ZIP,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as f:
    for item in entries:f.write(item,arcname=item.name)
with zipfile.ZipFile(ZIP) as f:assert f.testzip() is None,'ZIP dañado'
print('✅ Archivo ZIP guardado en Drive:',ZIP)
files.download(str(ZIP))


**Siguiente:** `12_CERDOS_Eficiencia_Computacional_GPU.ipynb`, usando GPU T4. El Notebook 12 no necesita los resultados de geometría para medir tiempos; ambos análisis son complementarios. Envía primero el ZIP 11 a ChatGPT para inspeccionar errores geométricos.